# Bronze — ingest_file
Moves new files from `Files/landing/<source>/` → `Files/bronze/<source>/` (immutable archive).

**Parameters injected by Airflow:** `source_config` (JSON string), `env`

In [ ]:
# Fabric Notebook — Bronze Layer: Ingest raw file into Bronze archive
# =====================================================================
# Triggered by: medallion_pipeline DAG → FabricRunItemOperator
# Parameters (injected by Airflow via notebook parameters cell):
#   source_config : JSON string of one row from sources.json
#   env           : "dev" | "prod"
#
# Responsibility:
#   1. List new files in Files/landing/<source_name>/
#   2. Move each file → Files/bronze/<source_name>/ (immutable archive)
#   3. Log the result to data_ops.pipeline_run_log
#   4. Exit with JSON summary for downstream Silver 1 task
#
# Bronze rule: NO transformation. Files land exactly as received.
#
# Lakehouse layout:
#   Files/landing/<source_name>/   ← clients upload here
#   Files/bronze/<source_name>/    ← pipeline archives here (never deleted)

import json
import sys
from pathlib import Path

### Shared lib

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

### Parameters

In [ ]:
try:
    source_config_str = source_config  # noqa: F821
    env = env  # noqa: F821
except NameError:
    source_config_str = "{}"
    env = "dev"

src: dict = json.loads(source_config_str)
source_name: str = src["source_name"]
bronze_path: str = src["bronze_path"]
landing_path: str = src.get("landing_path", f"landing/{source_name}/")
file_format: str = src["file_format"]  # csv | excel | parquet | json | avro

FORMAT_GLOB = {
    "csv":     "*.csv",
    "excel":   "*.xlsx",
    "parquet": "*.parquet",
    "json":    "*.json",
    "avro":    "*.avro",
}
file_glob = FORMAT_GLOB.get(file_format, f"*.{file_format}")

### Config

In [ ]:
config_path = Path(__file__).parent.parent.parent.parent / "config" / f"{env}.json"
cfg: dict = json.loads(config_path.read_text())

WORKSPACE   = cfg["workspace_name"]
LAKEHOUSE   = cfg["lakehouse"]
DW_ENDPOINT = cfg["gold_warehouse_sql_endpoint"]
DW_DATABASE = cfg["gold_warehouse"]

try:
    run_id = mssparkutils.notebook.run("_get_run_id")  # noqa: F821
except Exception:
    run_id = "local"

print(f"[bronze/ingest_file] source={source_name} env={env}")
print(f"  landing : Files/{landing_path}")
print(f"  archive : Files/{bronze_path}")

### 1. List already-processed files

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    records = gf.query_to_records(
        conn,
        f"SELECT file_name FROM data_ops.pipeline_run_log "
        f"WHERE source_name = '{source_name}' AND layer = 'bronze' AND status = 'success'"
    )
    processed_files = {r["file_name"] for r in records}

### 2. List new files in landing zone

In [ ]:
landing_files = gf.list_files(
    workspace_name=WORKSPACE,
    lakehouse_name=LAKEHOUSE,
    prefix=landing_path,
    pattern=file_glob,
)

new_files = [f for f in landing_files if f.split("/")[-1] not in processed_files]
print(f"  Landing: {len(landing_files)} files, {len(new_files)} new")

if not new_files:
    print("  No new files — nothing to do")
    mssparkutils.notebook.exit(json.dumps({"status": "no_new_files", "count": 0}))  # noqa: F821

### 3. Move each new file: landing/ → bronze/

In [ ]:
copied = []
errors = []

for src_abfss in new_files:
    file_name = src_abfss.split("/")[-1]
    dest_abfss = gf.onelake_abfss(
        WORKSPACE, LAKEHOUSE, f"Files/{bronze_path.rstrip('/')}/{file_name}"
    )

    try:
        # mssparkutils.fs.mv moves the file atomically (no separate delete needed)
        mssparkutils.fs.mv(src_abfss, dest_abfss, overwrite=True)  # noqa: F821
        print(f"  Moved {file_name} → {dest_abfss}")
        copied.append(file_name)

        with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
            gf.log_run(
                conn,
                source_name=source_name,
                layer="bronze",
                status="success",
                rows_processed=1,
                run_id=run_id,
            )

    except Exception as exc:
        print(f"  Failed to move {file_name}: {exc}", file=sys.stderr)
        errors.append({"file": file_name, "error": str(exc)})

        with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
            gf.log_run(
                conn,
                source_name=source_name,
                layer="bronze",
                status="failed",
                error_message=str(exc),
                run_id=run_id,
            )

if errors:
    raise RuntimeError(f"[bronze/ingest_file] {len(errors)} file(s) failed: {errors}")

### 4. Exit with summary

In [ ]:
result = {"status": "success", "source_name": source_name, "files_copied": copied}
print(f"  Result: {result}")
mssparkutils.notebook.exit(json.dumps(result))  # noqa: F821